# 01 — Exploratory Data Analysis

First look at the raw extract in `data/raw/`, before any cleaning. Goal here is to actually find the data-quality problems, not assume the source system is clean. Six specific issues are confirmed below; each one is fixed with a stated rule in `02_data_cleaning.ipynb`.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW = Path('data/raw')
orders = pd.read_csv(RAW / 'orders.csv')
customers = pd.read_csv(RAW / 'customers.csv')
products = pd.read_csv(RAW / 'products.csv')
regions = pd.read_csv(RAW / 'regions.csv')
returns = pd.read_csv(RAW / 'returns.csv')
marketing = pd.read_csv(RAW / 'marketing_spend.csv')

print('orders:', orders.shape)
print('customers:', customers.shape)
print('products:', products.shape)
print('returns:', returns.shape)
print('marketing_spend:', marketing.shape)

orders: (59584, 13)
customers: (12000, 5)
products: (148, 5)
returns: (4614, 5)
marketing_spend: (1296, 4)


In [2]:
orders.head()

,order_id,order_date,customer_id,product_id,region_id,channel,quantity,unit_price,discount_pct,revenue,cost,gross_profit,quality_flag
0,1,2025-09-02,9872.0,130,6.0,Social,1,40.63,0.000,40.63,24.82,15.81,NaN
1,2,2025-08-19,10280.0,77,6.0,Paid Search,11,15.02,0.230,165.22,85.91,79.31,NaN
2,3,2025-01-14,1695.0,9,5.0,Affiliate,2,485.26,0.049,970.52,829.02,141.50,NaN
3,4,2025-08-25,1224.0,31,5.0,Paid Search,2,108.13,0.130,216.26,116.82,99.44,NaN
4,5,2025-08-03,8021.0,7,1.0,Paid Search,1,534.25,0.092,534.25,424.38,109.87,NaN


In [3]:
orders.dtypes

order_id          int64
order_date          str
customer_id     float64
product_id        int64
region_id       float64
channel             str
quantity          int64
unit_price      float64
discount_pct    float64
revenue         float64
cost            float64
gross_profit    float64
quality_flag        str
dtype: object

## Issue 1 — Duplicate order rows

In [4]:
compare_cols = [c for c in orders.columns if c != 'order_id']
dupe_mask = orders.duplicated(subset=compare_cols, keep=False)
print(f'Rows involved in duplicate sets: {dupe_mask.sum()}')
orders[dupe_mask].sort_values(compare_cols).head(4)

Rows involved in duplicate sets: 918


,order_id,order_date,customer_id,product_id,region_id,channel,quantity,unit_price,discount_pct,revenue,cost,gross_profit,quality_flag
4105,4106,2023-06-22,2742.0,19,4.0,Email,2,126.34,0.089,252.68,211.60,41.08,NaN
52527,52528,2023-06-22,2742.0,19,4.0,Email,2,126.34,0.089,252.68,211.60,41.08,NaN
1614,1615,2023-07-19,9756.0,111,2.0,Organic,2,83.96,0.015,167.92,98.32,69.60,NaN
19842,19843,2023-07-19,9756.0,111,2.0,Organic,2,83.96,0.015,167.92,98.32,69.60,NaN


## Issue 2 — Missing `customer_id` (guest checkouts)

In [5]:
n_missing_cust = orders['customer_id'].isna().sum()
print(f'{n_missing_cust} orders ({n_missing_cust/len(orders)*100:.2f}%) have no customer_id.')

1847 orders (3.10%) have no customer_id.


## Issue 3 — Inconsistent channel labels

In [6]:
print('Unique channel labels in raw data:', orders["channel"].nunique())
orders['channel'].value_counts()

Unique channel labels in raw data: 19


channel
Paid Search    16433
Social         13346
Organic        10121
Email           5573
Affiliate       5531
Direct          4464
Paid_Search      568
PAID SEARCH      518
paid search      512
social           440
SOCIAL           390
ORGANIC          343
organic          317
email            200
AFFILIATE        187
affiliate        177
EMAIL            159
direct           154
DIRECT           151
Name: count, dtype: int64

These should collapse to 6 canonical channels (Paid Search, Social, Organic, Email, Affiliate, Direct) — a downstream marketing export merged in different casing/format conventions and it was never standardized before landing here.

## Issue 4 — Zero or negative `unit_price`

In [7]:
bad_price = orders[orders['unit_price'] <= 0]
print(f'{len(bad_price)} rows have unit_price <= 0.')
bad_price[['order_id','order_date','unit_price','revenue']].head()

187 rows have unit_price <= 0.


,order_id,order_date,unit_price,revenue
161,162,2024-05-30,-17.859023,-17.86
545,546,2025-04-03,0.000000,0.00
875,876,2025-03-24,-17.859023,-17.86
1288,1289,2024-02-10,0.000000,0.00
1299,1300,2025-10-22,-17.859023,-71.44


## Issue 5 — Missing `region_id`

In [8]:
n_missing_region = orders['region_id'].isna().sum()
print(f'{n_missing_region} orders ({n_missing_region/len(orders)*100:.2f}%) have no region_id.')

1430 orders (2.40%) have no region_id.


## Issue 6 — Refunds larger than the original order

In [9]:
check = returns.merge(orders[['order_id','revenue']], on='order_id', how='left')
over_refund = check[check['refund_amount'] > check['revenue']]
print(f'{len(over_refund)} return rows refund MORE than the order was worth.')
over_refund[['return_id','order_id','refund_amount','revenue']].head()

32 return rows refund MORE than the order was worth.


,return_id,order_id,refund_amount,revenue
94,95,1234,30.08,25.92
156,157,1952,276.40,240.06
159,160,2072,66.73,53.68
406,407,5104,328.25,282.64
856,857,11025,49.72,45.08


## Sanity check — does `gross_profit` actually equal `revenue - cost`?

In [10]:
mismatch = orders[(orders['unit_price'] > 0) & 
                   (( orders['revenue'] - orders['cost'] ).round(2) != orders['gross_profit'].round(2))]
print(f'{len(mismatch)} rows where gross_profit does not reconcile — none expected, confirms the ' 
      'generator computed this field correctly for valid rows.')

0 rows where gross_profit does not reconcile — none expected, confirms the generator computed this field correctly for valid rows.


## Summary of confirmed issues

| # | Issue | Rows affected |
|---|---|---|
| 1 | Duplicate order rows | see count above |
| 2 | Missing `customer_id` (guest checkout) | ~3% |
| 3 | Inconsistent channel label casing | 19 raw variants → 6 canonical |
| 4 | Zero/negative `unit_price` | 187 rows |
| 5 | Missing `region_id` | ~2.4% |
| 6 | Refund exceeds original order value | several dozen rows |

Each is addressed explicitly (not silently dropped) in `02_data_cleaning.ipynb`.